# Lineshape-parameter diagnostics — E791 Fit 2

This notebook isolates the failure mode seen when resonance mass/width parameters are released. It deliberately compares a dominant component, $\sigma\pi^+$ (Fit-2 fraction about 46%), with the weak $\rho(1450)\pi^+$ component (about 0.7%).

The historical E791 three-pion model used effective Blatt-Weisskopf radii of $3.0\,\mathrm{GeV}^{-1}$ for both the $D$ and the resonance factors. The current fitter uses its covariant angular convention, so this remains a Fit-2-based closure model rather than a bit-for-bit reconstruction of the historical code.

The diagnostics are hierarchical: first float only shape parameters, then the same resonance coefficient, then all coefficients. The same pseudo-data are reused throughout. We also compare 100k and 1M normalization samples to expose finite-MC normalization bias.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, Minimizer, NonResonant, Parameter,
    RealImag, Resonance, enable_x64, weighted_resample,
)

enable_x64()


## 1. Common Fit-2 model inputs


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma": (1.17, 205.7), "rho770": (1.0, 0.0),
    "NR": (0.48, 57.3), "f0_980": (0.43, 165.0),
    "f2_1270": (0.76, 57.3), "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r*np.cos(phase), r*np.sin(phase)

def internal_xy(name):
    r, phase = fit2_polar[name]
    if name == "NR":
        phase += 180.0  # project RBW sign convention relative to constant NR
    return polar_to_xy(r, phase)

truth_xy = {name: internal_xy(name) for name in fit2_polar}
resonance_data = {
    "sigma": (0.478, 0.324, 0),
    "rho770": (0.7693, 0.1502, 1),
    "f0_980": (0.975, 0.044, 0),
    "f2_1270": (1.275, 0.185, 2),
    "f0_1370": (1.434, 0.173, 0),
    "rho1450": (1.465, 0.310, 1),
}

for name, value in truth_xy.items():
    print(name, value)


## 2. Model builder for controlled hypotheses

This local helper exists only to keep the diagnostic cells readable. It does not change the package API.


In [ ]:
def build_model(dynamic=None, free_component_coefficients=(), free_all_coefficients=False):
    truth = {}
    coefficients = {}
    for name in truth_xy:
        x0, y0 = truth_xy[name]
        if name == "rho770":
            coefficients[name] = RealImag(1.0, 0.0)
            continue
        free = free_all_coefficients or name in free_component_coefficients
        if free:
            x = Parameter.coefficient(f"{name}.x", 0.0, owner=name, bounds=(-2.0, 2.0), step=0.01)
            y = Parameter.coefficient(f"{name}.y", 0.0, owner=name, bounds=(-2.0, 2.0), step=0.01)
            coefficients[name] = RealImag(x, y)
            truth[x.name], truth[y.name] = x0, y0
        else:
            coefficients[name] = RealImag(x0, y0)

    components = []
    for name in ("sigma", "rho770", "f0_980", "f2_1270", "f0_1370", "rho1450"):
        mass0, width0, spin = resonance_data[name]
        mass, width = mass0, width0
        if name == dynamic:
            mass = Parameter.dynamics(
                f"{name}.mass", mass0*0.95, owner=name,
                bounds=(max(0.20, mass0*0.70), mass0*1.30), step=0.002,
            )
            width = Parameter.dynamics(
                f"{name}.width", width0*1.15, owner=name,
                bounds=(max(0.01, width0*0.35), width0*2.0), step=0.003,
            )
            truth[mass.name], truth[width.name] = mass0, width0
        components.append(
            Resonance(
                name, (0,1), coefficients[name], mass=mass, width=width, spin=spin,
                resonance_radius=3.0, parent_radius=3.0,
            )
        )
    components.append(NonResonant(coefficients["NR"]))
    return DecayModel(channel, components), truth

truth_model, _ = build_model()


## 3. One common pseudo-data sample

Use a candidate pool much larger than the pseudo-data sample. This avoids the severe discrete-pool duplication that occurs when `N_POOL == N_DATA` and resampling is done with replacement.


In [ ]:
N_POOL = 1_000_000
N_DATA = 100_000

pool = truth_model.generate_phase_space(N_POOL, seed=2000)
truth_weight = pool.weights * truth_model.intensity(pool.as_dict())
data = weighted_resample(jax.random.key(791), pool, truth_weight, N_DATA, replace=True)

# One large normalization sample; the first 100k events form the small sample.
norm_large = truth_model.generate_phase_space(1_000_000, seed=2027)
norm_small = norm_large.take(jnp.arange(100_000))
print("data", data.size, "norm small", norm_small.size, "norm large", norm_large.size)


## 4. Fit helper and diagnostics


In [ ]:
def run_fit(label, model, truth, norm, n_starts=12):
    cache = model.prepare_cache(data, norm)
    def nll(values):
        intensity, normalization = cache.evaluate(values)
        return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)

    minimizer = Minimizer(nll, model.parameters)
    scan = minimizer.fit_multistart(
        n_starts=n_starts, seed=314159, include_default=False, simplex=True
    )
    result = scan.best
    print("\n" + label)
    print("valid =", result.valid, "NLL =", float(result.fval), "EDM =", float(result.fmin.edm))
    print("NLL(best)-NLL(truth) =", float(result.fval - nll(truth)))
    print(f"{'parameter':16s} {'truth':>11s} {'fit':>11s} {'error':>11s} {'pull':>10s}")
    for p in model.parameters:
        if p.fixed:
            continue
        fit = float(result.values[p.name])
        err = float(result.errors[p.name])
        pull = (truth[p.name]-fit)/err
        print(f"{p.name:16s} {truth[p.name]:11.6f} {fit:11.6f} {err:11.6f} {pull:10.3f}")
    return cache, nll, scan, result


## 5. Dominant $\sigma$: shape parameters only


In [ ]:
model_sigma_shape, truth_sigma_shape = build_model(dynamic="sigma")
res_sigma_shape = run_fit(
    "sigma mass/width only — 1M normalization",
    model_sigma_shape, truth_sigma_shape, norm_large,
)


## 6. $\sigma$: coefficient + mass + width


In [ ]:
model_sigma4, truth_sigma4 = build_model(
    dynamic="sigma", free_component_coefficients=("sigma",)
)
res_sigma4 = run_fit(
    "sigma x/y/mass/width — 1M normalization",
    model_sigma4, truth_sigma4, norm_large,
)


## 7. Full coefficient fit + $m_\sigma,\Gamma_\sigma$


In [ ]:
model_sigma_full, truth_sigma_full = build_model(
    dynamic="sigma", free_all_coefficients=True
)
res_sigma_full = run_fit(
    "all coefficients + sigma mass/width — 1M normalization",
    model_sigma_full, truth_sigma_full, norm_large,
    n_starts=20,
)


## 8. Weak $\rho(1450)$: coefficient + mass + width

This is the direct comparison with the difficult component.


In [ ]:
model_rho4, truth_rho4 = build_model(
    dynamic="rho1450", free_component_coefficients=("rho1450",)
)
res_rho4 = run_fit(
    "rho1450 x/y/mass/width — 1M normalization",
    model_rho4, truth_rho4, norm_large,
)


## 9. Full coefficient fit + $m_{\rho(1450)},\Gamma_{\rho(1450)}$


In [ ]:
model_rho_full, truth_rho_full = build_model(
    dynamic="rho1450", free_all_coefficients=True
)
res_rho_full = run_fit(
    "all coefficients + rho1450 mass/width — 1M normalization",
    model_rho_full, truth_rho_full, norm_large,
    n_starts=20,
)


## 10. Normalization-MC sensitivity: 100k versus 1M

The pseudo-data are identical. Any substantial movement of the fitted mass/width when only the integration sample size changes is integration error, not a statistical fluctuation of the data.


In [ ]:
_, _, _, rho_small = run_fit(
    "rho1450 x/y/mass/width — 100k normalization",
    model_rho4, truth_rho4, norm_small,
)
_, _, _, rho_large = res_rho4

for name in ("rho1450.mass", "rho1450.width"):
    small = float(rho_small.values[name])
    large = float(rho_large.values[name])
    print(f"{name:16s}: 100k={small:.6f}  1M={large:.6f}  shift={small-large:+.6f}")


## 11. Local NLL curvature at truth

A shallow profile for $\rho(1450)$ compared with $\sigma$ is direct evidence of weak identifiability. These scans keep all other coordinates at truth; they are not profiled scans, so they measure local raw sensitivity only.


In [ ]:
def scan_coordinate(model, truth, norm, name, grid):
    cache = model.prepare_cache(data, norm)
    def nll(values):
        intensity, normalization = cache.evaluate(values)
        return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)
    values = []
    for x in grid:
        point = dict(truth)
        point[name] = float(x)
        values.append(float(nll(point)))
    values = np.asarray(values)
    return values-values.min()

sigma_m_grid = np.linspace(0.40, 0.56, 60)
rho_m_grid = np.linspace(1.30, 1.60, 60)
sigma_dnll = scan_coordinate(model_sigma4, truth_sigma4, norm_large, "sigma.mass", sigma_m_grid)
rho_dnll = scan_coordinate(model_rho4, truth_rho4, norm_large, "rho1450.mass", rho_m_grid)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(sigma_m_grid, sigma_dnll, label=r"$m_\sigma$")
ax.plot(rho_m_grid, rho_dnll, label=r"$m_{\rho(1450)}$")
ax.axhline(0.5, linestyle="--", linewidth=1)
ax.set(xlabel="mass [GeV]", ylabel=r"$\Delta$NLL (others fixed)", ylim=(0,10))
ax.legend()
plt.show()
